# 02 — Building the customer support agent

> **Applied Labs** · **Domain**: AG · **Difficulty**: Advanced · **Reading time**: ~40 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/01_customer_support_agent/02_build.ipynb)

**Prerequisites**:
- [01_architecture.ipynb](./01_architecture.ipynb) — pipeline stages, knowledge base, intent routing, prompt design

**What you will learn**:
- How to build a complete `SupportAgent` class with intent classification, RAG, and response generation
- How sliding-window memory and entity extraction enable multi-turn conversations
- How to integrate simulated tools (order lookup) and route between tool use and RAG
- How confidence-based escalation prevents dangerous wrong answers
- How to run end-to-end demos showing the full pipeline in action

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "openai>=1.0.0" "sentence-transformers>=2.2.0" "numpy>=1.24.0"

import os
import re
import json
import time
import numpy as np
from dataclasses import dataclass, field
from typing import Optional
from sentence_transformers import SentenceTransformer
from openai import OpenAI

# Embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# OpenAI client — set OPENAI_API_KEY in your environment
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("Setup complete ✓")

## Section 1 — Knowledge base ingestion

Before the agent can answer questions, it needs a knowledge base. We create a `VectorStore` class that:
1. Accepts documents with category tags
2. Chunks documents into manageable pieces
3. Embeds chunks with sentence-transformers
4. Supports filtered and unfiltered similarity search

In [ ]:
# VectorStore: chunk, embed, and retrieve documents

class VectorStore:
    """Simple in-memory vector store with category-filtered search."""

    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self.chunks: list[dict] = []
        self.embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[dict[str, str]], max_chunk_chars: int = 300) -> None:
        """Chunk and embed a list of documents with 'category', 'title', 'content' keys."""
        for doc in documents:
            text = doc["content"]
            sentences = text.replace(". ", ".|").split("|")
            current = ""
            for sentence in sentences:
                if len(current) + len(sentence) > max_chunk_chars and current:
                    self.chunks.append({
                        "text": current.strip(),
                        "category": doc["category"],
                        "title": doc["title"],
                    })
                    current = sentence
                else:
                    current += (" " + sentence) if current else sentence
            if current.strip():
                self.chunks.append({
                    "text": current.strip(),
                    "category": doc["category"],
                    "title": doc["title"],
                })
        texts = [c["text"] for c in self.chunks]
        self.embeddings = self.model.encode(texts, show_progress_bar=False)

    def search(self, query: str, top_k: int = 3,
               category: str | None = None) -> list[dict]:
        """Return top-k chunks by cosine similarity, optionally filtered by category."""
        from sklearn.metrics.pairwise import cosine_similarity
        query_emb = self.model.encode([query])
        if category:
            indices = [i for i, c in enumerate(self.chunks) if c["category"] == category]
            if not indices:
                indices = list(range(len(self.chunks)))
        else:
            indices = list(range(len(self.chunks)))
        filtered_emb = self.embeddings[indices]
        scores = cosine_similarity(query_emb, filtered_emb)[0]
        top_local = np.argsort(scores)[::-1][:top_k]
        return [
            {**self.chunks[indices[i]], "score": float(scores[i])}
            for i in top_local
        ]

    def stats(self) -> dict[str, int]:
        """Return chunk count statistics by category."""
        from collections import Counter
        counts = Counter(c["category"] for c in self.chunks)
        return {"total": len(self.chunks), "by_category": dict(counts)}

In [ ]:
# Build the knowledge base with 15+ product docs, FAQs, and policies

DOCUMENTS: list[dict[str, str]] = [
    {"category": "product_info", "title": "Wireless Headphones Pro",
     "content": "The Wireless Headphones Pro feature 40-hour battery life, active noise cancellation, "
                "Bluetooth 5.3, and USB-C fast charging. Compatible with iOS, Android, and Windows. "
                "Available in black, white, and midnight blue. Weight: 250g. Driver size: 40mm."},
    {"category": "product_info", "title": "Portable Speaker Max",
     "content": "The Portable Speaker Max delivers 360-degree sound with 20W output. IPX7 waterproof "
                "rating, 12-hour battery, and built-in microphone for calls. Pairs with up to 3 devices "
                "simultaneously. Dimensions: 180mm x 75mm. Weight: 680g."},
    {"category": "product_info", "title": "USB-C Fast Charger",
     "content": "65W GaN USB-C charger with dual ports. Supports PD 3.0 and QC 4.0. Compatible with "
                "laptops, tablets, and phones. Foldable prongs for travel. Safety certified: UL, FCC, CE."},
    {"category": "product_info", "title": "Mechanical Keyboard Elite",
     "content": "Hot-swappable mechanical keyboard with RGB backlighting. Cherry MX Brown switches. "
                "N-key rollover, USB-C detachable cable. Aluminum frame with wrist rest included."},
    {"category": "product_info", "title": "Wireless Mouse Ergo",
     "content": "Ergonomic wireless mouse with 4000 DPI sensor. Bluetooth and 2.4GHz dual mode. "
                "6 programmable buttons, silent clicks. 90-day battery on single AA."},
    {"category": "general", "title": "FAQ: Account creation",
     "content": "To create an account, visit our website and click Sign Up. Enter your email, create "
                "a password (min 8 characters), and verify via email. Google and Apple sign-in supported."},
    {"category": "general", "title": "FAQ: Warranty",
     "content": "All products include a 1-year limited warranty covering manufacturing defects. Does not "
                "cover physical damage or unauthorized modifications. Contact support to file a claim."},
    {"category": "general", "title": "FAQ: International shipping",
     "content": "We ship to 45+ countries. International orders may incur customs duties (buyer's "
                "responsibility). Delivery takes 7-14 business days."},
    {"category": "order_status", "title": "Shipping: Standard",
     "content": "Standard shipping: 3-5 business days in the US. Orders before 2 PM EST ship same day. "
                "Free on orders over $50. Tracking number emailed within 24 hours."},
    {"category": "order_status", "title": "Shipping: Express",
     "content": "Express shipping: 1-2 business days for $12.99 (free over $100). Orders before 12 PM "
                "EST ship same day. Not available for PO boxes."},
    {"category": "order_status", "title": "Shipping: Delays",
     "content": "If your order is delayed, allow 2 extra business days. Common causes: weather, carrier "
                "volume, address holds. We reship or refund lost orders."},
    {"category": "billing", "title": "Returns: Standard",
     "content": "Items returnable within 30 days for full refund. Must be in original packaging, unused. "
                "Free return shipping with prepaid label. Refunds in 5-7 business days."},
    {"category": "billing", "title": "Returns: Defective",
     "content": "Defective items returnable within 90 days. No packaging required. We cover shipping. "
                "Replacements ship within 2 business days."},
    {"category": "billing", "title": "Billing: Duplicate charges",
     "content": "Duplicate charges often drop in 24 hours (pending authorizations). If persistent, contact "
                "support with order number. Refund issued within 3 business days."},
    {"category": "technical_support", "title": "Tech: Bluetooth pairing",
     "content": "Pairing steps: 1) Enable Bluetooth. 2) Hold power button 5 seconds (LED flashes blue). "
                "3) Select device in Bluetooth settings. 4) If failing, forget device and retry. "
                "Factory reset: hold power + volume down 10 seconds."},
    {"category": "technical_support", "title": "Tech: Battery issues",
     "content": "If not charging: 1) Try different cable. 2) Clean port with compressed air. 3) Ensure "
                "charger is 5V/1A minimum. 4) Try different outlet. If unresolved, contact warranty support."},
    {"category": "technical_support", "title": "Tech: Firmware updates",
     "content": "Download the TechGear companion app. Connect via Bluetooth, go to Settings > Firmware. "
                "Keep device above 50% charge. Updates take 3-5 minutes."},
]

# Build and populate the vector store
store = VectorStore(embed_model)
store.add_documents(DOCUMENTS)

stats = store.stats()
print(f"Knowledge base loaded: {stats['total']} chunks")
for cat, count in sorted(stats["by_category"].items()):
    print(f"  {cat}: {count} chunks")

## Section 2 — Support agent class

Now we assemble the full agent. The `SupportAgent` class orchestrates:
1. **Intent classification** via embedding centroids
2. **Document retrieval** via the VectorStore (filtered by intent)
3. **Prompt construction** with retrieved context
4. **LLM response generation** via OpenAI
5. **Confidence assessment** based on retrieval scores and response analysis

In [ ]:
# Full SupportAgent class

from sklearn.metrics.pairwise import cosine_similarity as cos_sim


class SupportAgent:
    """End-to-end customer support agent with RAG, intent routing, and escalation."""

    INTENT_EXAMPLES: dict[str, list[str]] = {
        "order_status": [
            "Where is my order?", "My package hasn't arrived",
            "Track my shipment", "When will my order be delivered?",
        ],
        "product_info": [
            "What are the specs?", "Is it waterproof?",
            "Battery life?", "Compatible with Mac?",
        ],
        "billing": [
            "I was charged twice", "Where is my refund?",
            "Return this item", "Wrong charge on my card",
        ],
        "technical_support": [
            "Won't connect to Bluetooth", "Not charging",
            "Firmware update", "Sound is distorted",
        ],
        "complaint": [
            "This is terrible", "Very unhappy",
            "Worst service ever", "Speak to a manager",
        ],
        "general": [
            "Create an account", "Warranty policy",
            "Ship internationally?", "Reset my password",
        ],
    }

    SYSTEM_PROMPT = (
        "You are a TechGear customer support agent. Be friendly, concise, and professional.\n"
        "Answer using ONLY the provided context. Cite sources as [Source: title].\n"
        "If context is insufficient, say you'll connect them with a specialist.\n"
        "Keep responses under 150 words. Lead with empathy if the customer is upset.\n"
        "Never make promises without policy backing. Never reveal internal processes."
    )

    def __init__(self, vector_store: VectorStore, openai_client: OpenAI,
                 embed_model: SentenceTransformer, llm_model: str = "gpt-4o-mini"):
        self.store = vector_store
        self.client = openai_client
        self.embed_model = embed_model
        self.llm_model = llm_model
        self.intent_centroids = self._build_centroids()
        self.escalation_threshold = 0.35

    def _build_centroids(self) -> dict[str, np.ndarray]:
        """Compute embedding centroids for each intent category."""
        centroids = {}
        for intent, examples in self.INTENT_EXAMPLES.items():
            embs = self.embed_model.encode(examples)
            centroids[intent] = np.mean(embs, axis=0)
        return centroids

    def classify_intent(self, message: str) -> tuple[str, float]:
        """Classify the intent of a user message using centroid similarity."""
        emb = self.embed_model.encode([message])
        best_intent, best_score = "general", 0.0
        for intent, centroid in self.intent_centroids.items():
            score = cos_sim(emb, centroid.reshape(1, -1))[0][0]
            if score > best_score:
                best_score = score
                best_intent = intent
        return best_intent, float(best_score)

    def retrieve(self, query: str, intent: str, top_k: int = 3) -> list[dict]:
        """Retrieve relevant knowledge chunks filtered by intent category."""
        return self.store.search(query, top_k=top_k, category=intent)

    def generate_response(self, query: str, context: list[dict],
                          history: list[dict] | None = None) -> str:
        """Generate a response using OpenAI with retrieved context."""
        context_text = "\n\n".join(
            f"[Source: {c['title']}] (relevance: {c['score']:.3f})\n{c['text']}"
            for c in context
        )
        history_text = "\n".join(
            f"{m['role'].upper()}: {m['content']}" for m in (history or [])
        ) or "(New conversation)"

        messages = [
            {"role": "system", "content": self.SYSTEM_PROMPT},
            {"role": "user", "content": (
                f"CONTEXT:\n{context_text}\n\n"
                f"HISTORY:\n{history_text}\n\n"
                f"CUSTOMER: {query}"
            )},
        ]
        response = self.client.chat.completions.create(
            model=self.llm_model, messages=messages,
            temperature=0.3, max_tokens=300,
        )
        return response.choices[0].message.content.strip()

    def assess_confidence(self, retrieval_scores: list[float], response: str) -> float:
        """Compute a confidence score based on retrieval quality and response hedging."""
        avg_retrieval = np.mean(retrieval_scores) if retrieval_scores else 0.0
        hedging_phrases = [
            "i'm not sure", "i don't have enough", "might be", "possibly",
            "i cannot confirm", "let me connect you", "specialist",
        ]
        hedge_count = sum(1 for p in hedging_phrases if p in response.lower())
        hedge_penalty = min(0.3, hedge_count * 0.1)
        return max(0.0, min(1.0, avg_retrieval - hedge_penalty))

    def handle_message(self, message: str,
                       history: list[dict] | None = None) -> dict:
        """Process a customer message through the full pipeline.

        Returns a dict with intent, confidence, response, escalated flag,
        and retrieved context.
        """
        intent, intent_conf = self.classify_intent(message)
        chunks = self.retrieve(message, intent)
        retrieval_scores = [c["score"] for c in chunks]
        response = self.generate_response(message, chunks, history)
        confidence = self.assess_confidence(retrieval_scores, response)
        escalated = confidence < self.escalation_threshold

        if escalated:
            response = (
                "I want to make sure you get the best help possible. "
                "Let me connect you with a specialist who can assist you further."
            )

        return {
            "intent": intent,
            "intent_confidence": intent_conf,
            "confidence": confidence,
            "escalated": escalated,
            "response": response,
            "retrieved_chunks": [c["text"][:80] + "..." for c in chunks],
        }


agent = SupportAgent(store, client, embed_model)
print(f"SupportAgent initialized with {len(agent.intent_centroids)} intent categories")
print(f"Escalation threshold: {agent.escalation_threshold}")

## Section 3 — Conversation memory

Single-turn responses miss critical context. A sliding-window memory manager:
- Keeps the last N turns to stay within token limits
- Extracts and accumulates entities across turns
- Provides the full context to the response generator

In [ ]:
# Conversation memory with sliding window and entity extraction

class ConversationMemory:
    """Sliding-window conversation memory with entity accumulation."""

    def __init__(self, max_turns: int = 10):
        self.max_turns = max_turns
        self.messages: list[dict[str, str]] = []
        self.entities: dict[str, str] = {}

    def add(self, role: str, content: str) -> None:
        """Add a message and extract entities from user messages."""
        self.messages.append({"role": role, "content": content})
        if len(self.messages) > self.max_turns * 2:
            self.messages = self.messages[-self.max_turns * 2:]
        if role == "user":
            self._extract_entities(content)

    def _extract_entities(self, text: str) -> None:
        """Extract order IDs, products, amounts, and emails from text."""
        order_match = re.search(r"#?(\d{4,})", text)
        if order_match:
            self.entities["order_id"] = order_match.group(1)
        products = ["headphones", "speaker", "charger", "keyboard", "mouse"]
        for p in products:
            if p in text.lower():
                self.entities["product"] = p
        amount_match = re.search(r"\$([\d,.]+)", text)
        if amount_match:
            self.entities["amount"] = amount_match.group(1)

    def get_history(self) -> list[dict[str, str]]:
        """Return the current conversation history."""
        return list(self.messages)

    def get_context_summary(self) -> str:
        """Return a summary of accumulated context."""
        parts = [f"Turns: {len(self.messages) // 2}"]
        if self.entities:
            parts.append(f"Entities: {self.entities}")
        return " | ".join(parts)


# Demonstrate multi-turn conversation with context accumulation
memory = ConversationMemory()
conversation = [
    "Hi, I bought the wireless headphones last week",
    "Order number is #45678 and I paid $89.99",
    "The left earpiece has no sound. Can I get a replacement?",
]

print("=== Multi-turn conversation with memory ===\n")
for i, msg in enumerate(conversation, 1):
    memory.add("user", msg)
    result = agent.handle_message(msg, memory.get_history())
    memory.add("assistant", result["response"])

    print(f"Turn {i}")
    print(f"  User: {msg}")
    print(f"  Intent: {result['intent']} (conf: {result['intent_confidence']:.3f})")
    print(f"  Context: {memory.get_context_summary()}")
    print(f"  Agent: {result['response'][:120]}...")
    print()

## Section 4 — Tool integration

Not everything can be answered from static knowledge. Order status queries need a **tool call** — a simulated database lookup. The agent must decide:
- **RAG path**: product info, policies, general questions → search knowledge base
- **Tool path**: order status with an order ID → call the order lookup tool

In [ ]:
# Simulated order database and tool router

ORDER_DATABASE: dict[str, dict] = {
    "45678": {"status": "shipped", "carrier": "UPS", "tracking": "1Z999AA10123456784",
              "estimated_delivery": "2025-01-20", "items": ["Wireless Headphones Pro"]},
    "51234": {"status": "processing", "carrier": None, "tracking": None,
              "estimated_delivery": "2025-01-22", "items": ["Mechanical Keyboard Elite"]},
    "39876": {"status": "delivered", "carrier": "FedEx", "tracking": "794644790132",
              "estimated_delivery": "2025-01-15", "items": ["USB-C Fast Charger", "Wireless Mouse Ergo"]},
    "62100": {"status": "cancelled", "carrier": None, "tracking": None,
              "estimated_delivery": None, "items": ["Portable Speaker Max"]},
    "78432": {"status": "shipped", "carrier": "USPS", "tracking": "9400111899223100012",
              "estimated_delivery": "2025-01-21", "items": ["Wireless Headphones Pro"]},
}


def lookup_order(order_id: str) -> dict | None:
    """Look up an order in the simulated database."""
    return ORDER_DATABASE.get(order_id)


def format_order_info(order: dict) -> str:
    """Format order data into a human-readable context string."""
    lines = [
        f"Order status: {order['status']}",
        f"Items: {', '.join(order['items'])}",
    ]
    if order["carrier"]:
        lines.append(f"Carrier: {order['carrier']}")
    if order["tracking"]:
        lines.append(f"Tracking: {order['tracking']}")
    if order["estimated_delivery"]:
        lines.append(f"Estimated delivery: {order['estimated_delivery']}")
    return "\n".join(lines)


class ToolRouter:
    """Decides whether to use a tool or RAG for a given query."""

    def __init__(self, agent: SupportAgent):
        self.agent = agent

    def route(self, message: str, entities: dict[str, str]) -> dict:
        """Route a message to either a tool or RAG, return the result."""
        intent, conf = self.agent.classify_intent(message)
        order_id = entities.get("order_id") or self._extract_order_id(message)

        if intent == "order_status" and order_id:
            order = lookup_order(order_id)
            if order:
                return {
                    "route": "tool",
                    "tool": "order_lookup",
                    "data": format_order_info(order),
                    "intent": intent,
                }
            return {
                "route": "tool",
                "tool": "order_lookup",
                "data": f"No order found with ID #{order_id}",
                "intent": intent,
            }
        return {
            "route": "rag",
            "tool": None,
            "data": None,
            "intent": intent,
        }

    def _extract_order_id(self, text: str) -> str | None:
        """Extract an order ID from the message text."""
        match = re.search(r"#?(\d{4,})", text)
        return match.group(1) if match else None


# Test the tool router
router = ToolRouter(agent)

test_messages = [
    ("Where is my order #45678?", {"order_id": "45678"}),
    ("Is the speaker waterproof?", {}),
    ("I need to track order #51234", {}),
    ("How do I return a defective item?", {}),
    ("What happened to order #99999?", {}),
]

print(f"{'Message':<40} {'Route':<6} {'Tool':<15} {'Data/Intent'}")
print("=" * 100)
for msg, entities in test_messages:
    result = router.route(msg, entities)
    data_preview = (result["data"] or result["intent"])[:40]
    tool = result["tool"] or "-"
    print(f"{msg:<40} {result['route']:<6} {tool:<15} {data_preview}")

## Section 5 — Confidence-based escalation

A support agent must know when it **doesn't know**. The confidence gate checks:
1. **Retrieval similarity** — are the retrieved chunks actually relevant?
2. **Response hedging** — does the generated response contain uncertainty markers?
3. **Topic sensitivity** — is this a high-risk category (billing, complaints)?

When confidence is low, the agent escalates to a human rather than risk a wrong answer.

In [ ]:
# Enhanced confidence gate with topic sensitivity

class ConfidenceGate:
    """Decides whether to serve a response or escalate to a human agent."""

    HIGH_RISK_INTENTS = {"billing", "complaint"}
    HEDGING_PHRASES = [
        "i'm not sure", "i don't have enough", "might be", "possibly",
        "i cannot confirm", "let me connect you", "specialist", "i think",
    ]

    def __init__(self, base_threshold: float = 0.35, risk_penalty: float = 0.1):
        self.base_threshold = base_threshold
        self.risk_penalty = risk_penalty

    def evaluate(self, intent: str, retrieval_scores: list[float],
                 response: str) -> dict[str, float | bool]:
        """Evaluate whether a response should be served or escalated.

        Returns confidence score, component scores, and escalation decision.
        """
        retrieval_conf = float(np.mean(retrieval_scores)) if retrieval_scores else 0.0
        hedge_count = sum(1 for p in self.HEDGING_PHRASES if p in response.lower())
        hedge_score = max(0.0, 1.0 - hedge_count * 0.15)
        risk_adjustment = self.risk_penalty if intent in self.HIGH_RISK_INTENTS else 0.0
        effective_threshold = self.base_threshold + risk_adjustment
        confidence = retrieval_conf * 0.6 + hedge_score * 0.4
        escalate = confidence < effective_threshold

        return {
            "confidence": round(confidence, 3),
            "retrieval_conf": round(retrieval_conf, 3),
            "hedge_score": round(hedge_score, 3),
            "effective_threshold": round(effective_threshold, 3),
            "escalated": escalate,
        }


gate = ConfidenceGate()

# Test scenarios: some should escalate, some should pass
scenarios = [
    ("product_info", [0.82, 0.75, 0.68],
     "The Wireless Headphones Pro has 40-hour battery life and active noise cancellation."),
    ("billing", [0.45, 0.38, 0.30],
     "I'm not sure about that charge. I think you might need to contact your bank."),
    ("order_status", [0.88, 0.81, 0.79],
     "Your order #45678 shipped via UPS and should arrive by January 20th."),
    ("complaint", [0.40, 0.35, 0.28],
     "I understand your frustration. Let me connect you with a specialist who can help."),
    ("technical_support", [0.72, 0.65, 0.60],
     "To fix Bluetooth pairing, hold the power button for 5 seconds until the LED flashes blue."),
]

print(f"{'Intent':<18} {'Conf':>5} {'Thresh':>6} {'Ret':>5} {'Hedge':>5} {'Decision':<10}")
print("=" * 65)
for intent, scores, response in scenarios:
    result = gate.evaluate(intent, scores, response)
    decision = "ESCALATE" if result["escalated"] else "SERVE"
    print(f"{intent:<18} {result['confidence']:>5.3f} {result['effective_threshold']:>6.3f} "
          f"{result['retrieval_conf']:>5.3f} {result['hedge_score']:>5.3f} {decision:<10}")

## Section 6 — End-to-end demo

Let's run 10 diverse queries through the full pipeline — intent classification, tool routing, RAG retrieval, response generation, and confidence gating — to see the complete system in action.

In [ ]:
# End-to-end pipeline with tool routing and confidence gating

def run_full_pipeline(message: str, memory: ConversationMemory,
                      agent: SupportAgent, router: ToolRouter,
                      gate: ConfidenceGate) -> dict:
    """Run a message through the full support pipeline."""
    memory.add("user", message)
    intent, intent_conf = agent.classify_intent(message)
    route_result = router.route(message, memory.entities)

    if route_result["route"] == "tool":
        # Tool path: inject tool data as context
        tool_context = [{"text": route_result["data"], "title": "Order System",
                         "category": intent, "score": 0.95}]
        response = agent.generate_response(message, tool_context, memory.get_history())
        retrieval_scores = [0.95]
    else:
        # RAG path: retrieve from knowledge base
        chunks = agent.retrieve(message, intent)
        response = agent.generate_response(message, chunks, memory.get_history())
        retrieval_scores = [c["score"] for c in chunks]

    gate_result = gate.evaluate(intent, retrieval_scores, response)

    if gate_result["escalated"]:
        response = ("I want to make sure you get the best help. "
                    "Let me connect you with a specialist.")

    memory.add("assistant", response)

    return {
        "intent": intent,
        "intent_confidence": round(intent_conf, 3),
        "route": route_result["route"],
        "tool": route_result["tool"],
        "confidence": gate_result["confidence"],
        "escalated": gate_result["escalated"],
        "response": response,
    }


# Run 10 diverse queries
test_queries = [
    "What's the battery life on the headphones?",
    "Where is my order #45678?",
    "I was charged twice for my keyboard purchase",
    "How do I pair my speaker with my iPhone?",
    "I want to return a defective charger I received",
    "Do you ship to Australia?",
    "My headphones won't charge at all",
    "Track order #51234 please",
    "This is the worst product I've ever bought. I want a manager NOW.",
    "How do I update the firmware on my speaker?",
]

print("=" * 80)
print("END-TO-END PIPELINE DEMO")
print("=" * 80)

for i, query in enumerate(test_queries, 1):
    mem = ConversationMemory()
    result = run_full_pipeline(query, mem, agent, router, gate)
    status = "🔴 ESCALATED" if result["escalated"] else "🟢 SERVED"
    print(f"\n[{i}] {query}")
    print(f"    Intent: {result['intent']} ({result['intent_confidence']:.3f})")
    print(f"    Route: {result['route']}" + (f" → {result['tool']}" if result["tool"] else ""))
    print(f"    Confidence: {result['confidence']:.3f} | {status}")
    print(f"    Response: {result['response'][:150]}{'...' if len(result['response']) > 150 else ''}")

## Exercises

1. **Add a new tool**: Implement a `check_warranty_status(product, purchase_date)` tool. Update the `ToolRouter` to call it when the intent is `technical_support` and a product entity is present. Test with 3 warranty-related queries.

2. **Improve entity extraction**: Extend `ConversationMemory._extract_entities` to handle dates ("last Tuesday", "January 15th"), phone numbers, and multi-word product names ("wireless headphones pro"). Verify with a 5-turn conversation.

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | A VectorStore class encapsulates chunking, embedding, and filtered search in one reusable component |
| 2 | The SupportAgent orchestrates intent → retrieval → generation → confidence in a single `handle_message` call |
| 3 | Sliding-window memory with entity accumulation enables coherent multi-turn conversations |
| 4 | Tool routing lets the agent call structured APIs (order lookup) when RAG alone is insufficient |
| 5 | Confidence gating with topic sensitivity prevents high-risk wrong answers in billing and complaints |
| 6 | End-to-end testing with diverse queries reveals routing, retrieval, and escalation behavior |

## What's Next

- **Next**: [03_evaluate.ipynb](./03_evaluate.ipynb) — Evaluate the agent with LLM-as-judge scoring, hallucination detection, and cost analysis
- **Related**: [01_architecture.ipynb](./01_architecture.ipynb) — Review the architecture decisions behind each pipeline stage

## References & Further Reading

1. [Schick et al., "Toolformer: Language Models Can Teach Themselves to Use Tools," 2023](https://arxiv.org/abs/2302.04761) — Foundation for tool-augmented LLM agents
2. [OpenAI, "Function Calling and Other API Updates," 2023](https://openai.com/index/function-calling-and-other-api-updates/) — The API pattern used for tool integration in production agents